# Smart Meter Load Forecasting and Anomaly Detection

**Portfolio project · UCI Electricity Load Diagrams 2011–2014**

This notebook analyzes 15-minute electricity consumption for up to 370 clients, compares a seasonal-naive baseline with a pooled Random Forest, and flags unusually large one-step forecast errors. All splits follow time order. The dataset has no anomaly labels, so flags are candidates for review rather than confirmed faults.

**Before running:** download `LD2011_2014.txt` from the [UCI repository](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014) into `data/LD2011_2014.txt`. The source file is large and is excluded from version control.

## 1. Setup and configuration

The model is trained only on the earliest 72% of dates. The next 8% calibrates anomaly thresholds; the final 20% is the chronological test period.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
DATA_PATH = Path("data/LD2011_2014.txt")
FREQ = "15min"
STEPS_PER_DAY = 96
LOOKBACK = 672  # one week of 15-minute readings
MAX_TRAIN_ROWS = 500_000
MAX_CALIBRATION_ROWS = 400_000
ANOMALY_QUANTILE = 0.99

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
print(f"Dataset path: {DATA_PATH.resolve()}")

C:\Users\kumar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.sparse import issparse


## 2. Load and clean the readings

The UCI file is semicolon-delimited. Values marked `?` become missing. Duplicate timestamps are averaged, the timeline is reindexed to 15-minute intervals, and only interior gaps of up to four intervals (one hour) are interpolated.

In [ ]:
def interpolate_short_gaps(frame: pd.DataFrame, max_gap: int = 4):
    """Fill only complete, interior missing runs no longer than max_gap."""
    candidate = frame.interpolate(method="time", limit_direction="both", limit_area="inside")
    cleaned = frame.copy()
    for column in frame.columns:
        missing = frame[column].isna()
        groups = missing.ne(missing.shift(fill_value=False)).cumsum()
        run_lengths = missing.groupby(groups).transform("sum")
        fillable = missing & run_lengths.le(max_gap)
        cleaned.loc[fillable, column] = candidate.loc[fillable, column]
    return cleaned


def load_electricity_data(path: Path = DATA_PATH):
    """Load UCI's wide timestamp-by-client file and apply limited gap repair."""
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Download LD2011_2014.txt from the UCI page "
            "and place it in the data/ directory; see README.md."
        )

    raw = pd.read_csv(
        path,
        sep=";",
        index_col=0,
        parse_dates=[0],
        na_values=["?", ""],
        low_memory=False,
    )
    raw.index = pd.DatetimeIndex(raw.index, name="timestamp")
    raw = raw.apply(pd.to_numeric, errors="coerce").astype("float32")
    raw = raw.sort_index()
    duplicate_count = int(raw.index.duplicated(keep=False).sum())
    if duplicate_count:
        raw = raw.groupby(level=0, sort=True).mean()

    if raw.empty:
        raise ValueError("The source file contains no readings.")
    expected = pd.date_range(raw.index.min(), raw.index.max(), freq=FREQ, name="timestamp")
    irregular_count = int((raw.index.to_series().diff().dropna() != pd.Timedelta(FREQ)).sum())
    raw = raw.reindex(expected)
    missing_before = raw.isna().mean().rename("fraction_missing_before_interpolation")
    cleaned = interpolate_short_gaps(raw, max_gap=4)
    missing_after = cleaned.isna().mean().rename("fraction_missing_after_interpolation")
    quality = pd.concat([missing_before, missing_after], axis=1)

    print(f"Rows: {len(cleaned):,} | clients: {cleaned.shape[1]} | interval: {FREQ}")
    print(f"Duplicate source timestamps: {duplicate_count:,} | irregular source intervals: {irregular_count:,}")
    print(f"Range: {cleaned.index.min()} to {cleaned.index.max()}")
    return cleaned, quality


consumption, data_quality = load_electricity_data()
display(data_quality.sort_values(data_quality.columns[0], ascending=False).head(10))
display(consumption.iloc[:5, :5])

## 3. Explore daily and weekly patterns

Each curve averages over timestamps first and clients second, so the profile describes the typical client rather than allowing high-volume clients to dominate.

In [ ]:
hourly_profile = consumption.groupby(consumption.index.hour).mean().mean(axis=1)
weekday_profile = consumption.groupby(consumption.index.dayofweek).mean().mean(axis=1)
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(hourly_profile.index, hourly_profile.values, marker="o", color="#176B87")
axes[0].set(title="Average consumption by hour", xlabel="Hour of day", ylabel="Mean consumption (kW)")
axes[0].set_xticks(range(0, 24, 2))
axes[1].plot(range(7), weekday_profile.reindex(range(7)).values, marker="o", color="#B85C38")
axes[1].set(title="Average consumption by weekday", xlabel="Day of week", ylabel="Mean consumption (kW)")
axes[1].set_xticks(range(7), weekday_names, rotation=25, ha="right")
fig.tight_layout()
plt.show()

print("Hourly profile range:", round(float(hourly_profile.min()), 3), "to", round(float(hourly_profile.max()), 3), "kW")
print("Weekday profile range:", round(float(weekday_profile.min()), 3), "to", round(float(weekday_profile.max()), 3), "kW")

## 4. Chronological split and feature helpers

The feature row for timestamp *t* uses readings strictly before *t*. A row is eligible only when the target and required recent history are present. For recursive forecasts, predicted values are appended to the history one step at a time.

In [ ]:
def chronological_boundaries(n_timestamps: int):
    validation_start = int(n_timestamps * 0.72)
    test_start = int(n_timestamps * 0.80)
    if validation_start <= LOOKBACK or test_start - validation_start < STEPS_PER_DAY:
        raise ValueError("Not enough timestamps for the configured train/validation/test splits.")
    return validation_start, test_start


def valid_target_times(values: np.ndarray, client_idx: int, start: int, stop: int):
    """Target indices with a complete 96-step rolling window and a valid weekly lag."""
    start = max(start, LOOKBACK)
    stop = min(stop, len(values))
    if stop <= start:
        return np.empty(0, dtype=np.int32)
    finite = np.isfinite(values[:, client_idx])
    bad_prefix = np.r_[0, np.cumsum(~finite, dtype=np.int64)]
    times = np.arange(start, stop, dtype=np.int32)
    recent_ok = (bad_prefix[times] - bad_prefix[times - STEPS_PER_DAY]) == 0
    weekly_lag_ok = finite[times - LOOKBACK]
    target_ok = finite[times]
    return times[recent_ok & weekly_lag_ok & target_ok]


def sample_rows(values, start, stop, cap, rng):
    """Sample roughly equally per client and spread samples across the time range."""
    n_clients = values.shape[1]
    quota = max(1, cap // n_clients)
    all_times, all_clients = [], []
    for client_idx in range(n_clients):
        candidates = valid_target_times(values, client_idx, start, stop)
        take = min(quota, len(candidates))
        if take:
            selected = np.sort(rng.choice(candidates, size=take, replace=False))
            all_times.append(selected)
            all_clients.append(np.full(take, client_idx, dtype=np.int16))
    if not all_times:
        raise ValueError("No complete training rows found. Check input timestamps and missingness.")
    return np.concatenate(all_times), np.concatenate(all_clients)


NUMERIC_FEATURES = [
    "lag_1", "lag_4", "lag_96", "lag_672", "mean_last_4", "mean_last_96",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
]


def make_features(values, timestamps, target_times, client_indices, client_names):
    """Build one-step supervised features for the supplied (time, client) pairs."""
    t = np.asarray(target_times, dtype=np.int32)
    c = np.asarray(client_indices, dtype=np.int32)
    if len(t) != len(c):
        raise ValueError("target_times and client_indices must have equal lengths")
    if not len(t):
        return pd.DataFrame(columns=NUMERIC_FEATURES + ["client_id"])
    ts = timestamps.take(t)
    hour = ts.hour.to_numpy() + ts.minute.to_numpy() / 60.0
    weekday = ts.dayofweek.to_numpy() + hour / 24.0
    recent4 = values[t[:, None] - np.arange(1, 5), c[:, None]]
    recent96 = values[t[:, None] - np.arange(1, STEPS_PER_DAY + 1), c[:, None]]
    X = pd.DataFrame({
        "lag_1": values[t - 1, c],
        "lag_4": values[t - 4, c],
        "lag_96": values[t - STEPS_PER_DAY, c],
        "lag_672": values[t - LOOKBACK, c],
        "mean_last_4": np.nanmean(recent4, axis=1),
        "mean_last_96": np.nanmean(recent96, axis=1),
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "weekday_sin": np.sin(2 * np.pi * weekday / 7),
        "weekday_cos": np.cos(2 * np.pi * weekday / 7),
        "client_id": np.asarray(client_names, dtype=object)[c],
    })
    return X


def build_forecaster(random_state=RANDOM_STATE):
    preprocess = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", NUMERIC_FEATURES),
            ("client", OneHotEncoder(handle_unknown="ignore"), ["client_id"]),
        ],
        sparse_threshold=1.0,
    )
    forest = RandomForestRegressor(
        n_estimators=120,
        max_depth=16,
        max_features=0.7,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=random_state,
    )
    return Pipeline([("features", preprocess), ("forest", forest)])


def mae_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return np.nan, np.nan
    errors = y_true[valid] - y_pred[valid]
    return float(np.mean(np.abs(errors))), float(np.sqrt(np.mean(errors ** 2)))


# Small hand-checkable checks for core indexing and metric behavior.
toy_values = np.tile(np.arange(800, dtype=np.float32)[:, None], (1, 2))
toy_index = pd.date_range("2020-01-01", periods=800, freq=FREQ)
toy_X = make_features(toy_values, toy_index, np.array([672, 673]), np.array([0, 1]), ["A", "B"])
assert len(toy_X) == 2 and toy_X.loc[0, "lag_1"] == 671 and toy_X.loc[0, "lag_672"] == 0
assert np.allclose(mae_rmse([0, 1], [0, 2]), (0.5, np.sqrt(0.5)))
print("Feature indexing and hand-calculated MAE/RMSE checks passed.")

## 5. Fit the pooled Random Forest

The development period is divided into an early training sample (72%) and a later validation interval (8%). Validation rows calibrate anomaly thresholds; the final 20% remains untouched until evaluation. One-hot client identity lets the shared forest learn client-specific splits without treating client names as ordered numbers.

In [ ]:
values = consumption.to_numpy(dtype=np.float32, copy=True)
timestamps = consumption.index
client_names = consumption.columns.astype(str).to_numpy()
validation_start, test_start = chronological_boundaries(len(timestamps))

rng = np.random.default_rng(RANDOM_STATE)
train_t, train_c = sample_rows(values, LOOKBACK, validation_start, MAX_TRAIN_ROWS, rng)
X_train = make_features(values, timestamps, train_t, train_c, client_names)
y_train = values[train_t, train_c]
model = build_forecaster()
model.fit(X_train, y_train)

print(f"Training rows: {len(y_train):,} (cap {MAX_TRAIN_ROWS:,})")
print(f"Train through: {timestamps[validation_start - 1]}")
print(f"Validation: {timestamps[validation_start]} through {timestamps[test_start - 1]}")
print(f"Test begins: {timestamps[test_start]}")

## 6. Calibrate client-specific anomaly thresholds

The 99th percentile of each client's absolute one-step validation error is the alert threshold. Thresholds are learned before the test period and applied only to test readings.

In [ ]:
cal_t, cal_c = sample_rows(values, validation_start, test_start, MAX_CALIBRATION_ROWS, rng)
X_cal = make_features(values, timestamps, cal_t, cal_c, client_names)
cal_pred = model.predict(X_cal)
cal_errors = np.abs(values[cal_t, cal_c] - cal_pred)
calibration = pd.DataFrame({"client_id": client_names[cal_c], "abs_error": cal_errors})
thresholds = calibration.groupby("client_id")["abs_error"].quantile(ANOMALY_QUANTILE)
global_threshold = float(calibration["abs_error"].quantile(ANOMALY_QUANTILE))
thresholds = thresholds.reindex(client_names).fillna(global_threshold)
print(f"Calibrated thresholds for {thresholds.notna().sum()} clients; global fallback: {global_threshold:.3f} kW")
display(thresholds.rename("99th_percentile_abs_error_kW").describe().to_frame())

## 7. Recursive 24-hour forecasts and chronological evaluation

Forecasts are generated every seven days in the held-out period. Each 96-step model path feeds its own predictions back as lags. The seasonal-naive forecast uses the actual previous day's same quarter-hour readings. Metrics are computed for both methods on the same valid forecast points, pooled and summarized by client.

In [ ]:
def recursive_forecast(model, values, timestamps, origin, client_names):
    """Forecast 96 points after origin; ineligible clients remain NaN."""
    n_clients = values.shape[1]
    first_history = origin - LOOKBACK + 1
    if first_history < 0:
        return np.full((STEPS_PER_DAY, n_clients), np.nan, dtype=np.float32)
    history = values[first_history:origin + 1].copy()
    # A 96-step path uses the latest 96 observations and the prior week's corresponding 96 lags.
    eligible = np.isfinite(history[:STEPS_PER_DAY]).all(axis=0) & np.isfinite(history[-STEPS_PER_DAY:]).all(axis=0)
    predictions = np.full((STEPS_PER_DAY, n_clients), np.nan, dtype=np.float32)
    if not eligible.any():
        return predictions

    for step in range(STEPS_PER_DAY):
        target_time = origin + step + 1
        if target_time >= len(timestamps):
            break
        tstamp = timestamps[target_time]
        hour = tstamp.hour + tstamp.minute / 60.0
        weekday = tstamp.dayofweek + hour / 24.0
        clients = np.flatnonzero(eligible)
        recent = history[:, eligible]
        X_step = pd.DataFrame({
            "lag_1": recent[-1],
            "lag_4": recent[-4],
            "lag_96": recent[-STEPS_PER_DAY],
            "lag_672": recent[-LOOKBACK],
            "mean_last_4": recent[-4:].mean(axis=0),
            "mean_last_96": recent[-STEPS_PER_DAY:].mean(axis=0),
            "hour_sin": np.full(len(clients), np.sin(2 * np.pi * hour / 24)),
            "hour_cos": np.full(len(clients), np.cos(2 * np.pi * hour / 24)),
            "weekday_sin": np.full(len(clients), np.sin(2 * np.pi * weekday / 7)),
            "weekday_cos": np.full(len(clients), np.cos(2 * np.pi * weekday / 7)),
            "client_id": client_names[clients],
        })
        step_pred = model.predict(X_step).astype(np.float32)
        predictions[step, clients] = step_pred
        history = np.vstack([history[1:], np.full((1, n_clients), np.nan, dtype=np.float32)])
        history[-1, clients] = step_pred
    return predictions


def recursive_forecasts_many(model, values, timestamps, origins, client_names):
    """Batch weekly origins so each horizon needs one model call, not one call per origin."""
    origins = np.asarray(origins, dtype=np.int32)
    if not len(origins):
        return np.empty((0, STEPS_PER_DAY, values.shape[1]), dtype=np.float32)
    histories = np.stack([values[o - LOOKBACK + 1:o + 1] for o in origins])
    n_origins, _, n_clients = histories.shape
    eligible = (
        np.isfinite(histories[:, :STEPS_PER_DAY, :]).all(axis=1)
        & np.isfinite(histories[:, -STEPS_PER_DAY:, :]).all(axis=1)
    )
    paths = np.full((n_origins, STEPS_PER_DAY, n_clients), np.nan, dtype=np.float32)

    for step in range(STEPS_PER_DAY):
        origin_rows, client_rows = np.where(eligible)
        if not len(origin_rows):
            break
        history_rows = histories[origin_rows][np.arange(len(origin_rows)), :, client_rows]
        if step:
            prior_steps = np.arange(step, dtype=np.int32)
            prior_predictions = paths[origin_rows[:, None], prior_steps[None, :], client_rows[:, None]]
        else:
            prior_predictions = np.empty((len(origin_rows), 0), dtype=np.float32)

        target_times = origins[origin_rows] + step + 1
        ts = timestamps.take(target_times)
        hour = ts.hour.to_numpy() + ts.minute.to_numpy() / 60.0
        weekday = ts.dayofweek.to_numpy() + hour / 24.0
        n_actual_4 = max(0, 4 - step)
        n_actual_96 = STEPS_PER_DAY - step
        actual_4_sum = history_rows[:, -n_actual_4:].sum(axis=1) if n_actual_4 else np.zeros(len(origin_rows))
        actual_96_sum = history_rows[:, -n_actual_96:].sum(axis=1)
        prior_4_sum = prior_predictions[:, -4:].sum(axis=1)
        prior_96_sum = prior_predictions.sum(axis=1)
        if step == 0:
            lag_1 = history_rows[:, -1]
            lag_4 = history_rows[:, -4]
        else:
            lag_1 = prior_predictions[:, -1]
            lag_4 = history_rows[:, -4 + step] if step < 4 else prior_predictions[:, -4]

        X_step = pd.DataFrame({
            "lag_1": lag_1,
            "lag_4": lag_4,
            "lag_96": history_rows[:, -STEPS_PER_DAY + step],
            "lag_672": history_rows[:, step],
            "mean_last_4": (actual_4_sum + prior_4_sum) / 4.0,
            "mean_last_96": (actual_96_sum + prior_96_sum) / STEPS_PER_DAY,
            "hour_sin": np.sin(2 * np.pi * hour / 24),
            "hour_cos": np.cos(2 * np.pi * hour / 24),
            "weekday_sin": np.sin(2 * np.pi * weekday / 7),
            "weekday_cos": np.cos(2 * np.pi * weekday / 7),
            "client_id": client_names[client_rows],
        })
        step_predictions = model.predict(X_step).astype(np.float32)
        paths[origin_rows, step, client_rows] = step_predictions
    return paths


def evaluate_weekly_origins(model, values, timestamps, test_start, client_names):
    model_errors, baseline_errors = [[] for _ in client_names], [[] for _ in client_names]
    overall_model_true, overall_model_pred = [], []
    overall_base_true, overall_base_pred = [], []
    first_forecast = None
    origins = np.asarray(list(range(test_start - 1, len(timestamps) - STEPS_PER_DAY, 7 * STEPS_PER_DAY)), dtype=np.int32)
    model_paths = recursive_forecasts_many(model, values, timestamps, origins, client_names)
    for origin_idx, origin in enumerate(origins):
        model_path = model_paths[origin_idx]
        target_slice = slice(origin + 1, origin + 1 + STEPS_PER_DAY)
        actual = values[target_slice]
        seasonal = values[origin + 1 - STEPS_PER_DAY:origin + 1]
        if first_forecast is None:
            first_forecast = (origin, model_path.copy(), actual.copy(), seasonal.copy())
        for client_idx in range(values.shape[1]):
            y = actual[:, client_idx]
            p_model = model_path[:, client_idx]
            p_base = seasonal[:, client_idx]
            # Compare both forecasts on the same valid observations.
            common = np.isfinite(y) & np.isfinite(p_model) & np.isfinite(p_base)
            if common.any():
                model_errors[client_idx].extend((y[common] - p_model[common]).tolist())
                baseline_errors[client_idx].extend((y[common] - p_base[common]).tolist())
                overall_model_true.extend(y[common].tolist())
                overall_model_pred.extend(p_model[common].tolist())
                overall_base_true.extend(y[common].tolist())
                overall_base_pred.extend(p_base[common].tolist())

    if first_forecast is None:
        raise ValueError("The test interval is too short for a 24-hour forecast.")
    pooled_model = mae_rmse(overall_model_true, overall_model_pred)
    pooled_baseline = mae_rmse(overall_base_true, overall_base_pred)
    rows = []
    for idx, client in enumerate(client_names):
        m = np.asarray(model_errors[idx], dtype=float)
        b = np.asarray(baseline_errors[idx], dtype=float)
        rows.append({
            "client_id": client,
            "model_MAE_kW": np.mean(np.abs(m)) if len(m) else np.nan,
            "model_RMSE_kW": np.sqrt(np.mean(m ** 2)) if len(m) else np.nan,
            "baseline_MAE_kW": np.mean(np.abs(b)) if len(b) else np.nan,
            "baseline_RMSE_kW": np.sqrt(np.mean(b ** 2)) if len(b) else np.nan,
            "model_points": len(m),
        })
    return pooled_model, pooled_baseline, pd.DataFrame(rows), first_forecast


model_metrics, baseline_metrics, per_client_metrics, first_forecast = evaluate_weekly_origins(
    model, values, timestamps, test_start, client_names
)
comparison = pd.DataFrame(
    [model_metrics, baseline_metrics],
    index=["Random Forest (recursive)", "Seasonal naive (previous day)"],
    columns=["MAE_kW", "RMSE_kW"],
)
print("Pooled holdout metrics from weekly 24-hour forecast origins:")
display(comparison)
print("Per-client metric summary:")
display(per_client_metrics.drop(columns="client_id").describe().T)
print(f"Clients with at least one valid model forecast point: {(per_client_metrics['model_points'] > 0).sum()} / {len(client_names)}")

### Example 24-hour path

The chart shows the first held-out weekly origin. A client without a complete recent week remains unavailable for that origin and is counted as missing coverage.

In [ ]:
origin, example_model, example_actual, example_baseline = first_forecast
valid_clients = np.flatnonzero(np.isfinite(example_model).any(axis=0) & np.isfinite(example_actual).any(axis=0))
if len(valid_clients):
    chosen = valid_clients[0]
    future_times = timestamps[origin + 1:origin + 1 + STEPS_PER_DAY]
    plt.figure(figsize=(12, 4.5))
    plt.plot(future_times, example_actual[:, chosen], label="Observed", color="#222222", linewidth=1.5)
    plt.plot(future_times, example_baseline[:, chosen], label="Previous-day baseline", alpha=0.8)
    plt.plot(future_times, example_model[:, chosen], label="Random Forest", alpha=0.9)
    plt.title(f"24-hour holdout forecast · {client_names[chosen]}")
    plt.ylabel("Consumption (kW)")
    plt.xlabel("Timestamp")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No client had a complete recent history at the first holdout origin.")

## 8. Flag unusual one-step errors in the test period

This pass evaluates each valid test reading with the fixed model using only its preceding history. It stores flagged readings and counts, avoiding a large table of every ordinary observation.

In [ ]:
def flag_test_anomalies(model, values, timestamps, start, thresholds, client_names):
    flagged_parts = []
    # Compute each client's valid target indices once; daily re-scans were quadratic in test duration.
    valid_times_by_client = [
        valid_target_times(values, client_idx, start, len(timestamps))
        for client_idx in range(values.shape[1])
    ]
    counts = pd.Series(0, index=client_names, dtype="int64", name="flagged_readings")
    for block_start in range(start, len(timestamps), STEPS_PER_DAY):
        block_stop = min(block_start + STEPS_PER_DAY, len(timestamps))
        target_times, client_indices = [], []
        for client_idx, available_times in enumerate(valid_times_by_client):
            left = np.searchsorted(available_times, block_start, side="left")
            right = np.searchsorted(available_times, block_stop, side="left")
            valid_t = available_times[left:right]
            if len(valid_t):
                target_times.append(valid_t)
                client_indices.append(np.full(len(valid_t), client_idx, dtype=np.int16))
        if not target_times:
            continue
        t = np.concatenate(target_times)
        c = np.concatenate(client_indices)
        X = make_features(values, timestamps, t, c, client_names)
        pred = model.predict(X)
        actual = values[t, c]
        abs_error = np.abs(actual - pred)
        limit = thresholds.reindex(client_names[c]).to_numpy()
        is_flagged = abs_error > limit
        if is_flagged.any():
            part = pd.DataFrame({
                "timestamp": timestamps.take(t[is_flagged]),
                "client_id": client_names[c[is_flagged]],
                "observed_kW": actual[is_flagged],
                "predicted_kW": pred[is_flagged],
                "absolute_error_kW": abs_error[is_flagged],
                "threshold_kW": limit[is_flagged],
            })
            flagged_parts.append(part)
            counts = counts.add(part["client_id"].value_counts(), fill_value=0).astype("int64")
    flags = pd.concat(flagged_parts, ignore_index=True) if flagged_parts else pd.DataFrame(
        columns=["timestamp", "client_id", "observed_kW", "predicted_kW", "absolute_error_kW", "threshold_kW"]
    )
    return flags, counts


anomaly_flags, anomaly_counts = flag_test_anomalies(
    model, values, timestamps, test_start, thresholds, client_names
)
print(f"Flagged {len(anomaly_flags):,} unusual readings in the holdout.")
print(f"Flagged clients: {(anomaly_counts > 0).sum()} / {len(client_names)}")
display(anomaly_flags.sort_values("absolute_error_kW", ascending=False).head(15))
display(anomaly_counts.sort_values(ascending=False).head(10).to_frame())

## 9. Results and limitations

Use the metric comparison and client-level distribution to discuss whether the model improves on the seasonal baseline. Forecasts are recursive, so errors can accumulate over the day. The 99th-percentile rule will flag roughly the most extreme validation errors by construction; it does not establish that a reading is a fault. Clients with long missing spans may have fewer training examples, unavailable forecast origins, or no calibrated threshold.

In [ ]:
best = comparison["MAE_kW"].idxmin()
print(f"Lower pooled MAE on this holdout: {best}")
print("Test interval:", timestamps[test_start], "through", timestamps[-1])
print("Anomaly flags are screening indicators; validate against operational context before action.")